# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/octane21/flyrank-internship-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window



*   one row : one content per day for one client
*   table : fact_content_daily_peformance, dim_content
*   time window : 2025/01/01 - 2026/06-30





## 2. Fields: feature / label / context / excluded

*   feature : content_ages, days_since_update, impression_90_days, gsc_data_available, ga4_data_available_ clicks_90_days
*   label : future_decline

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

print("Secret successfully loaded!")

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN {HF_TOKEN})")  # accept the gate in-browser first, then paste a READ token
rel = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    'dim_clients':f"read_parquet('{rel}/dim_clients.parquet')",
    'dim_content':f"read_parquet('{rel}/dim_content.parquet')",
    'fact_content_daily_performance':f"read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')",
    'fact_content_query_90d':f"read_parquet('{rel}/fact_content_query_90d.parquet')",
}

MONTH_START = '2026-01-01'
MONTH_END = '2026-03-31'

Secret successfully loaded!


In [4]:
# Grain
schema = con.sql(f"""SELECT * FROM {TABLES['fact_content_daily_performance']} LIMIT 1""").df().dtypes
date_content_client = con.sql(f"""SELECT COUNT(DISTINCT report_date) as report_date_total, COUNT(DISTINCT client_hash_id) as client_total, COUNT(DISTINCT content_hash_id) as content_total FROM {TABLES['fact_content_daily_performance']} WHERE report_date >= DATE '{MONTH_START}' AND report_date <= DATE'{MONTH_END}'""").df()
duplicate = con.sql(f"""SELECT
            report_date,
            client_hash_id,
            content_hash_id,
            COUNT(*) as n
            FROM {TABLES['fact_content_daily_performance']}
            WHERE report_date >= DATE '{MONTH_START}' AND
            report_date <= DATE'{MONTH_END}'
            GROUP BY
            report_date,
            client_hash_id,
            content_hash_id
            HAVING n > 1 """).df()

print(schema,"\n")
print(date_content_client.to_string(index=False),"\n")
print(f"Duplicates = {len(duplicate)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

report_date                 datetime64[us]
client_hash_id                      object
content_hash_id                     object
client_has_gsc                        bool
client_has_ga4                        bool
gsc_data_available                    bool
ga4_data_available                    bool
gsc_impressions                      int64
gsc_clicks                           int64
gsc_sum_position                     int64
gsc_avg_position                   float64
ga4_pageviews                        int64
ga4_sessions                         int64
ga4_users                            int64
ga4_engaged_sessions                 int64
ga4_total_engagement_sec             int64
sessions_organic                     int64
sessions_direct                      int64
sessions_referral                    int64
sessions_social                      int64
sessions_paid                        int64
sessions_ai                          int64
ai_chatgpt                           int64
ai_perplexi

In [5]:
# Slice's row count and date span

count_row_date_span = con.sql(f"""SELECT
                                  COUNT(DISTINCT report_date) as report_date_total,
                                  COUNT(DISTINCT client_hash_id) as client_total,
                                  COUNT(DISTINCT content_hash_id) as content_total,
                                  MIN(report_date) as start_date,
                                  MAX(report_date) as end_date
                                  FROM
                                  {TABLES['fact_content_daily_performance']}
                                  WHERE
                                  report_date >= DATE '{MONTH_START}' AND
                                  report_date <= DATE'{MONTH_END}'""").df()


count_row_date_span

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date_total,client_total,content_total,start_date,end_date
0,90,59,349411,2026-01-01,2026-03-31


In [6]:
# Availability
row_content_gsc_ga4 = con.sql(f"""SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT content_hash_id) AS total_content,
            COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_data_available,
            COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_data_available,
            COUNT(*) FILTER (WHERE ga4_data_available IS TRUE AND gsc_data_available IS TRUE) AS ga4_gsc_data_available,
            FROM {TABLES['fact_content_daily_performance']} WHERE report_date>= DATE '{MONTH_START}' AND report_date <= DATE '{MONTH_END}';""").df()

content_90_days_gsc_ga4 = con.sql(f"""
    SELECT
        COUNT(*) AS content_90_days_gsc_ga4
    FROM (
        SELECT
            content_hash_id
        FROM {TABLES['fact_content_daily_performance']}
        WHERE report_date >= DATE '{MONTH_START}'
          AND report_date <= DATE '{MONTH_END}'
        GROUP BY content_hash_id
        HAVING COUNT(DISTINCT report_date) FILTER (
            WHERE gsc_data_available IS TRUE
              AND ga4_data_available IS TRUE
        ) >= 90
    ) AS content
""").df()

print(row_content_gsc_ga4.to_string(index=False))
print(content_90_days_gsc_ga4.to_string(index=False))
con.sql(f"""
    SELECT
        content_hash_id,
        MIN(report_date) as start_date_gsc_ga4,
        MAX(report_date) as end_date_gsc_ga4,
        COUNT(DISTINCT report_date) as range_gsc_ga4
    FROM (
        SELECT
            content_hash_id,
            report_date
        FROM {TABLES['fact_content_daily_performance']}
        WHERE report_date >= DATE '{MONTH_START}'
          AND report_date <= DATE '{MONTH_END}'
          AND gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE
   )
   GROUP BY content_hash_id
   ORDER BY range_gsc_ga4 DESC
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  total_content  gsc_data_available  ga4_data_available  ga4_gsc_data_available
   25087303         349411             8629987              675073                  589286
 content_90_days_gsc_ga4
                      27


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,start_date_gsc_ga4,end_date_gsc_ga4,range_gsc_ga4
0,content_e8a52cf3d5988c07,2026-01-01,2026-03-31,90
1,content_40baa8f1016f5742,2026-01-01,2026-03-31,90
2,content_6416361da197d254,2026-01-01,2026-03-31,90
3,content_fa84f5976d5fe3c1,2026-01-01,2026-03-31,90
4,content_ddd9373c30db5df6,2026-01-01,2026-03-31,90
...,...,...,...,...
68969,content_9da614f34790b18d,2026-03-30,2026-03-30,1
68970,content_2129accfb3de2233,2026-03-30,2026-03-30,1
68971,content_32ffd8e6a54278fb,2026-03-30,2026-03-30,1
68972,content_c05e5eee23aa748c,2026-03-30,2026-03-30,1


In [7]:
con.sql(f"""
    SELECT
        content_hash_id,
        MIN(report_date) as start_date_gsc_ga4,
        MAX(report_date) as end_date_gsc_ga4,
        COUNT(DISTINCT report_date) as range_gsc_ga4
    FROM (
        SELECT
            content_hash_id,
            report_date
        FROM {TABLES['fact_content_daily_performance']}
        WHERE report_date >= DATE '{MONTH_START}'
          AND report_date <= DATE '{MONTH_END}'
          AND gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE
   )
   GROUP BY content_hash_id
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,start_date_gsc_ga4,end_date_gsc_ga4,range_gsc_ga4
0,content_6a167adf0f285e1c,2026-01-01,2026-03-28,12
1,content_b2c8677f822e152a,2026-01-01,2026-03-31,34
2,content_3d4b913ae2cd04d1,2026-01-01,2026-03-29,35
3,content_2324c6bfae2be0c8,2026-01-01,2026-03-30,23
4,content_7d2ca0aec3326bfd,2026-01-01,2026-03-31,58
...,...,...,...,...
68969,content_32058aa8a2e4f4fb,2026-03-31,2026-03-31,1
68970,content_526be944d717ab76,2026-03-31,2026-03-31,1
68971,content_20c613ae83bad2ca,2026-03-31,2026-03-31,1
68972,content_d4f53cf222510f09,2026-03-31,2026-03-31,1


In [8]:
con.sql(f"""SELECT DISTINCT content_hash_id FROM (SELECT
            content_hash_id,
            gsc_data_available,
            ga4_data_available
            FROM {TABLES['fact_content_daily_performance']} WHERE report_date>= DATE '{MONTH_START}' AND report_date <= DATE '{MONTH_END}'
            GROUP BY
            content_hash_id,
            gsc_data_available,
            ga4_data_available);""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id
0,content_5bbd1b5ea9f4d15c
1,content_17ff914d33201392
2,content_19f818ed91e960ca
3,content_a1347678bacc57fd
4,content_f0f6a030c428f851
...,...
349406,content_9df1db0fc345a6c7
349407,content_f68eb2d3eb73fc7a
349408,content_6f9d5e123291a6fc
349409,content_3217015d75a5f329


In [9]:
con.sql(f"""
    SELECT COUNT(*) AS content_90_days_age
    FROM (
        SELECT
            content_hash_id
        FROM {TABLES['fact_content_daily_performance']}
        WHERE report_date >= DATE '{MONTH_START}'
          AND report_date <= DATE '{MONTH_END}'
        GROUP BY content_hash_id
        HAVING COUNT(DISTINCT report_date) >= 90
    ) AS content
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────┐
│ content_90_days_age │
│        int64        │
├─────────────────────┤
│              200079 │
└─────────────────────┘

In [10]:
fact_content_daily_90_days_hv_gs_ga4

NameError: name 'fact_content_daily_90_days_hv_gs_ga4' is not defined

In [ ]:
con.sql(f"SELECT COUNT(*) FROM {TABLES['fact_content_daily_performance']}  WHERE report_date >= DATE '2026-01-01' AND report_date <= DATE '{MONTH_END}'")


In [ ]:
fact_content_daily_90_days_hv_gs_ga4

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.